In [2]:
##############################################
##############################################
### TO GET ACCESS TO FILE IN GOOGLE COLAB ###
##############################################
##############################################

# To get access to the data files from github in Google Colab
#!git clone https://github.com/math869c/graph_representation_st457.git

# set the folder to get access to the data
#import os
#os.chdir('/content') # to avoid error if rerun
#os.chdir('./graph_representation_st457')

import sys
sys.path.append("/Users/stephwenninger/Desktop/LSE/ST457/graph_representation_st457")

In [2]:
# installation of Pytorch Geometric. This may take up to 30 minutes!!!

#!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-${TORCH}.html
#!pip install -q torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}.html
#!pip install -q git+https://github.com/pyg-team/pytorch_geometric.git

# install DGL
#!pip install dgl
#!pip install numpy==1.26.4

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torch.optim as optim
import json

# load from other folders
from helper_functions_steph import *
from model_classes_steph import *

# Load data for GAT+RotatE

In [2]:
# Stock prices
open_prices_interp = pd.read_csv('data_folder/open_prices_interp.csv', index_col=0)
# into numpy
x = open_prices_interp.to_numpy()
tickers_with_data = open_prices_interp.columns

# Load graph data
with open("data_folder/firm_industry.json", "r") as f:
    firm_industry_dict = json.load(f)
A = np.load("data_folder/adjacency_matrix.npy")
adj_matrix = torch.tensor(A, dtype=torch.float32)

# make training data
batch_size =32
X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(x,
                                                                                                        batch_size =batch_size,
                                                                                                        flatten_data = False, # Should be True for LSTM, False for GTC and GAT
                                                                                                        flatten_time_features=True # Should be True for GAT, False for LSTM and GTC
                                                                                                        )


torch.Size([984, 460, 40])
torch.Size([984, 460])


# Train GAT+RotatE model

In [ ]:
# build_edges is imported from helper_functions.py.
# It converts an adjacency tensor [N, N, R] into edge_index and edge_type for the RotatE loss.


In [3]:
# Tune GAT+RotatE and compare against the same model with lambda_rotate = 0.
# lambda_rotate = 0 is the plain forecasting GAT baseline: no RotatE graph loss is used.
from itertools import product
import copy

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# I did a relatively small grid search, otherwise this takes ages.
search_space = {
    'c_hidden': [16, 32],
    'emb_dim': [8, 16],
    'num_heads': [2, 4],
    'lr': [1e-3],
    'lambda_rotate': [0.0, 0.001, 0.01, 0.05, 0.1],
    'corr_threshold': [0.5],
}

num_epochs = 8
patience = 2
input_dim = X_train.shape[-1]


def make_adj_matrix(corr_threshold):
    corr = adj_matrix[:, :, 2]
    corr_bin = (corr.abs() > corr_threshold).float()
    corr_bin.fill_diagonal_(1.0)
    return torch.stack([adj_matrix[:, :, 0], adj_matrix[:, :, 1], corr_bin], dim=-1)


def trial_label(lambda_rotate):
    return 'GAT_lambda0' if lambda_rotate == 0 else 'GAT_RotatE'


def build_model(params, num_relations):
    return GAT_RotatE(
        c_in=input_dim,
        c_hidden=params['c_hidden'],
        emb_dim=params['emb_dim'],
        num_relations=num_relations,
        num_heads=params['num_heads'],
    )


results = []
best_by_group = {
    'GAT_lambda0': {'best_val_mse': float('inf')},
    'GAT_RotatE': {'best_val_mse': float('inf')},
}

keys = list(search_space.keys())
param_grid = [dict(zip(keys, values)) for values in product(*(search_space[k] for k in keys))]
print(f'Running {len(param_grid)} hyperparameter trials')

for trial_idx, params in enumerate(param_grid, start=1):
    # GAT hidden size must divide cleanly across attention heads.
    if params['c_hidden'] % params['num_heads'] != 0:
        continue

    torch.manual_seed(seed)
    np.random.seed(seed)

    A_loop = make_adj_matrix(params['corr_threshold'])
    edge_index, edge_type = build_edges(A_loop)
    num_relations = A_loop.shape[-1]

    model = build_model(params, num_relations)
    optimizer = optim.Adam(model.parameters(), lr=params['lr'])

    best_val_mse = float('inf')
    best_state_dict = None
    best_epoch = 0
    patience_counter = 0
    history = []

    print(f"Trial {trial_idx:03d}/{len(param_grid)} | {params}")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch_GAT_RotatE(
            model,
            train_loader,
            A_loop,
            edge_index,
            edge_type,
            optimizer,
            lambda_rotate=params['lambda_rotate'],
        )
        val_mse = evaluate_GAT_RotatE(model, val_loader, A_loop)
        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_mse': val_mse})

        print(f"  Epoch {epoch:02d} | train loss {train_loss:.6f} | val MSE {val_mse:.6f}")

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state_dict = copy.deepcopy(model.state_dict())
            best_epoch = epoch
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print('  Early stopping')
                break

    group = trial_label(params['lambda_rotate'])
    result = {
        **params,
        'model_type': group,
        'best_epoch': best_epoch,
        'best_val_mse': best_val_mse,
    }
    results.append(result)

    if best_val_mse < best_by_group[group]['best_val_mse']:
        best_by_group[group] = {
            'best_val_mse': best_val_mse,
            'params': params.copy(),
            'state_dict': best_state_dict,
            'A_loop': A_loop.clone(),
            'history': history,
        }

results_df = pd.DataFrame(results).sort_values('best_val_mse').reset_index(drop=True)
display(results_df)

# Evaluate the best lambda=0 model and the best positive-lambda model on the held-out test set.
final_results = {}
for group, best in best_by_group.items():
    if 'params' not in best:
        continue

    params = best['params']
    A_loop = best['A_loop']
    model = build_model(params, A_loop.shape[-1])
    model.load_state_dict(best['state_dict'])

    pred_test = predict_GAT_RotatE(model, test_loader, A_loop)
    mse_by_firm = np.mean((y_test - pred_test.numpy()) ** 2, axis=0)

    final_results[group] = {
        'params': params,
        'val_mse': best['best_val_mse'],
        'test_mse': mse_by_firm.mean(),
        'mse_by_firm': mse_by_firm,
        'pred': pred_test,
        'model': model,
    }

comparison_df = pd.DataFrame([
    {
        'model_type': group,
        'val_mse': result['val_mse'],
        'test_mse': result['test_mse'],
        **result['params'],
    }
    for group, result in final_results.items()
]).sort_values('val_mse').reset_index(drop=True)

display(comparison_df)


Running 40 hyperparameter trials
Trial 001/40 | {'c_hidden': 16, 'emb_dim': 8, 'num_heads': 2, 'lr': 0.001, 'lambda_rotate': 0.0, 'corr_threshold': 0.5}
  Epoch 01 | train loss 2.009763 | val MSE 5.157231
  Epoch 02 | train loss 1.352981 | val MSE 4.100386
  Epoch 03 | train loss 1.220217 | val MSE 3.745712
  Epoch 04 | train loss 1.155608 | val MSE 3.381724
  Epoch 05 | train loss 1.117663 | val MSE 3.108621
  Epoch 06 | train loss 1.088516 | val MSE 2.983594
  Epoch 07 | train loss 1.070900 | val MSE 2.906424
  Epoch 08 | train loss 1.061225 | val MSE 2.848940
Trial 002/40 | {'c_hidden': 16, 'emb_dim': 8, 'num_heads': 2, 'lr': 0.001, 'lambda_rotate': 0.001, 'corr_threshold': 0.5}
  Epoch 01 | train loss 2.223769 | val MSE 5.123282
  Epoch 02 | train loss 1.475045 | val MSE 4.091750
  Epoch 03 | train loss 1.318485 | val MSE 3.508441
  Epoch 04 | train loss 1.234945 | val MSE 3.204591
  Epoch 05 | train loss 1.184691 | val MSE 3.026431
  Epoch 06 | train loss 1.151191 | val MSE 2.9311

,c_hidden,emb_dim,num_heads,lr,lambda_rotate,corr_threshold,model_type,best_epoch,best_val_mse
0,32,16,2,0.001,0.050,0.5,GAT_RotatE,8,2.650262
1,32,16,2,0.001,0.100,0.5,GAT_RotatE,8,2.652833
2,32,16,2,0.001,0.010,0.5,GAT_RotatE,8,2.660345
3,32,16,4,0.001,0.050,0.5,GAT_RotatE,8,2.672581
4,32,16,4,0.001,0.010,0.5,GAT_RotatE,8,2.673520
5,32,16,4,0.001,0.100,0.5,GAT_RotatE,8,2.675487
6,16,16,4,0.001,0.001,0.5,GAT_RotatE,8,2.676316
7,16,16,4,0.001,0.010,0.5,GAT_RotatE,8,2.677014
8,16,16,4,0.001,0.050,0.5,GAT_RotatE,8,2.685505
9,16,16,4,0.001,0.100,0.5,GAT_RotatE,8,2.694394


,model_type,val_mse,test_mse,c_hidden,emb_dim,num_heads,lr,lambda_rotate,corr_threshold
0,GAT_RotatE,2.650262,1.578559,32,16,2,0.001,0.05,0.5
1,GAT_lambda0,2.722945,1.607638,16,16,4,0.001,0.00,0.5


In [ ]:
MSE_dict = {
    key: value['mse_by_firm']
    for key, value in final_results.items()
}


In [ ]:
print_box_plots(MSE_dict)
